# 01/08 优化：梯度下降家族 — 学习 Notebook

**学习状态：进行中（2026-11-04），正式测验尚未开始。**

官方英文原文：[08-optimization/docs/en.md](../../../../../../phases/01-math-foundations/08-optimization/docs/en.md)
课程原始代码：[08-optimization/code](../../../../../../phases/01-math-foundations/08-optimization/code/)

本课沿用上一课的学习方式：学生独立作答、自己编写代码；导师逐题验收，实际执行数值测试，并保留原始答案与订正过程。

**官方课程目标**：从零实现 Gradient Descent、SGD with Momentum、Adam；使用 Rosenbrock 函数比较收敛；掌握 mini-batch、学习率调度、凸与非凸问题和鞍点。

**课程节奏**：问题与概念 → Vanilla GD → Momentum → Adam → Rosenbrock 数值实验 → PyTorch 对照 → 四项官方练习 → 三道正式后测。原始 quiz.json 中还有两道前测；正式后测前不披露正确选项。

## 第 1 讲：训练为何需要优化器？

给定参数 $w$ 和损失函数 $L(w)$，目标是：

$$\min_w L(w).$$

在可微位置，梯度 $\nabla L(w)$ 指向局部一阶增长最快的方向。最基本的**梯度下降**使用更新：

$$\boxed{w_{t+1}=w_t-\eta\nabla L(w_t)},\quad\eta>0.$$

负号保证沿负梯度方向迈步，$\eta$ 称为学习率。对于固定方向 $d=-\nabla L(w)$，有方向导数 $\nabla L(w)^\top d=-\|\nabla L(w)\|^2\le0$；但**有限步长的损失并非总会下降**，这要取决于函数局部曲率与学习率。

从本课开始，每种优化器都要区分：更新公式、使用哪些历史状态、每步计算成本、收敛速度和稳定性。用相同初始点与可比较的预算做实验，不直接把某次运行的现象概括为普遍结论。

## 第 1 讲 · 第一题：手算一次梯度下降（等待学生回答）

这是理解检查，**不是**官方测验。

给定：

$$L(w)=(w-3)^2,\quad w_0=5,\quad\eta=0.1.$$

请依次计算：

1. 初始梯度 $L'(w_0)$；
2. 更新后的参数 $w_1=w_0-\eta L'(w_0)$；
3. 更新前的损失 $L(w_0)$ 与更新后的损失 $L(w_1)$，判断是否下降；
4. 用一句话说明为什么更新公式要减去梯度，而不是加上梯度。

导师在学生回答后给出逐项验收与可运行代码，然后进入本课原始前测及 Rosenbrock 函数的梯度推导。

In [ ]:
# 01/08：代码实验从下次学生提交开始。
# 学生先手算一轮梯度下降；教师将逐项验收。


## 后续官方 Exercises 与 Post Quiz（未开始）

官方四道练习（依据英文原文，不提前填入结果）：

1. **Learning rate sweep**：Rosenbrock 上 Vanilla GD 的学习率取 0.0001、0.0005、0.001、0.005、0.01，运行 5000 步并记录最终 loss。
2. **Momentum comparison**：动量分别取 0、0.5、0.9、0.99，追踪完整 loss 曲线、收敛及振荡。
3. **Saddle point escape**：在 $f(x,y)=x^2-y^2$、初始点 $(0.01,0.01)$ 上对比 GD、Momentum、Adam。
4. **Exponential learning-rate decay**：给 GD 加入 $\eta_t=\eta_0\cdot0.999^t$，与固定学习率的 Rosenbrock 结果比较。

官方 post quiz：3 题，完成上面实验后逐题进行并登记到仓库根目录 LEARNING.md。

## 第 1 讲：第一轮梯度下降手算（学生作答与批改）

原始回答：

> 1. 梯度是4 2.w_1是4.6 3.L(w_0)=4, L(w_1) = 2.56

**数值 3/3 正确。**

设 $L(w)=(w-3)^2$，从 $w_0=5$、$\eta=0.1$ 开始：

$$L'(w)=2(w-3),\qquad L'(5)=4.$$
$$w_1=5-0.1\cdot4=4.6.$$
$$L(w_0)=4,\qquad L(w_1)=2.56.$$

损失从 4 降至 2.56。原题还要求说明**为什么沿负梯度方向而非正梯度方向**，学生本轮没有回答；该概念仍需后续理解检查，不算已验收。

第 1 讲后续：在 $L'(w_0)=4>0$ 的位置，向较小 $w$ 方向移动会使局部损失下降；一般情况下，梯度和负梯度的方向导数相反。对于较大有限学习率，并不保证损失每次都下降。

## 课程资料同步说明

学习者希望将上游项目中文译本放到个人仓库对应镜像目录，并从远端拉取至本地 WSL。

本课中文机器译本已经从上游 `translations` 分支复制到：

    learning-artifacts/cn/phases/01-math-foundations/08-optimization/docs/zh.md

**质量标记：未经逐段人工校对的中文机器译文**，上游有明显技术术语误译。本课以官方英文 docs/en.md 为准进行数学与代码校核。

学生此前已克隆 fork 到 WSL，分支是 my-progress。建议先检查本地工作区，再执行：

    cd ~/ai-engineering-from-scratch
    git status --short
    git fetch origin
    git switch my-progress
    git pull --ff-only origin my-progress

若本地 my-progress 不存在，可 `git switch --track origin/my-progress`。本次同步与前序对话中 GitHub 操作都在 fork 的 my-progress 分支，不影响 main。

继续教学时请学生补答：为什么梯度下降沿负梯度方向，而不是正梯度方向？

## 本课课程资源完整镜像与持久规则（已验收）

用户明确要求：每次进入新课，先把官方整个课程目录的**全部内容**镜像到对应中文学习目录，并把此规则写入下次能够读取的持久化项目说明，避免只复制中文讲义。

已在 `my-progress` 分支完成的文件清单：

- 原始官方文件 5/5：`docs/en.md`、`code/optimizers.py`、`code/main.jl`、`quiz.json`、`outputs/prompt-optimizer-guide.md`；均复用官方 Git blob，路径与 SHA 完全一致。
- 中文与个人文件：`docs/zh.md`（上游机器译本，质量待校对）、`quiz.zh.json`（题序、阶段和答案下标与官方一致）、`outputs/prompt-optimizer-guide.zh.md`、`code/learn.ipynb`（保留原有内容）。
- 补充入口：本课 `README.md`、`source-manifest.json`。

已写入仓库持久规则：根目录 `AGENTS.md` 指向 `learning-artifacts/cn/AGENTS.md`，该文件详细规定整课镜像、中文参考、SHA 比对、保护原始学生解答、逐题验收及完成条件；根目录 `LEARNING.md` 和中文目录 `README.md` 也有入口。

已读取 API 目录清单验证全部官方原始文件的 Git blob SHA；已核对中文版 quiz 的 5 道题在 `stage` 与 `correct` 字段上与原版完全一致。Notebook 保持原先 7 个单元后新增本条记录。

后续教学尚待学生补答：为什么梯度下降沿负梯度方向？本课尚未正式完成。

## 第 2 讲：为什么沿负梯度方向，以及 Rosenbrock 函数

第一讲学生已独立算得 $L(w)=(w-3)^2$ 在 $w_0=5$、$\eta=0.1$ 时梯度 4、更新后 $w_1=4.6$、损失由 4 降至 2.56。

一阶泰勒展开：

$$L(\boldsymbol\theta+\Delta\boldsymbol\theta)\approx L(\boldsymbol\theta)+\nabla L(\boldsymbol\theta)^\top\Delta\boldsymbol\theta.$$

对于 $\Delta\boldsymbol\theta=-\eta\nabla L(\boldsymbol\theta)$，

$$L(\boldsymbol\theta-\eta\nabla L)\approx L(\boldsymbol\theta)-\eta\|\nabla L\|_2^2.$$

负梯度方向的方向导数为负（当梯度非零）；足够小的步长可局部降低可微函数损失，但有限步长过大时不能保证下降。

本课权威英文源文件中的测试函数是：

$$f(x,y)=(1-x)^2+100(y-x^2)^2.$$

全局最小值为 $f(1,1)=0$，其附近的狭窄曲线谷地会给优化器带来迂回和振荡问题。

### 学生下一题（待回答）

独立推导 $\partial f/\partial x$ 和 $\partial f/\partial y$，说明对 x 求偏导时如何使用链式法则。**不要提前填写标准导数答案**。学生提交后与解析梯度及有限差分测试交叉验证，之后再让学生自己实现 `rosenbrock_gradient(params)`。

本课知识和代码以后均对照同目录的原始 `docs/en.md`、`code/optimizers.py` 和官方中文学习材料；继续遵循 `learning-artifacts/cn/AGENTS.md` 中完整镜像与逐题验收规则。

## 01/08 · Rosenbrock 偏导验收

学生独立提交：

\[
\frac{\partial f}{\partial x}=-2(1-x)+200(y-x^2)(-2x),\qquad
\frac{\partial f}{\partial y}=200(y-x^2).
\]

**判定：两个偏导均正确。** 化简后：

\[
\nabla f(x,y)=
\begin{pmatrix}
-2(1-x)-400x(y-x^2)\\
200(y-x^2)
\end{pmatrix}.
\]

已在独立 Python 环境实际运行中心差分检查，6 个测试点乘以 2 个偏导共 **12/12 通过**，点为 (-1,1)、(1,1)、(0,0)、(1.4,2.2)、(-0.3,0.8)、(2,-3)，差分步长 1e-6。下个代码单元提供复现用的检查，未把教师程序冒充学生代码。

## 本课中文教材精译说明

已完整重译同课文档 zh.md，保留与英文版一致的 24 个标题、17 个代码及图示块和 4 个外部参考网址。对原文容易造成误解的优化理论陈述以译注注明适用条件。旧的低质量机器译本已替换，英文原文与个人学习记录未覆盖。

**下一步待学生完成：** 自主将偏导表达式写成 Python 函数 rosenbrock_gradient(params)，返回两个偏导组成的列表；教师随后进行实际运行与数值差分验收。

In [ ]:
import math
def f(x, y):
    return (1-x)**2 + 100*(y-x*x)**2
def grad(x, y):
    return (-2*(1-x)+200*(y-x*x)*(-2*x), 200*(y-x*x))
points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in points:
    gx,gy = grad(x,y)
    dx = (f(x+h,y)-f(x-h,y))/(2*h)
    dy = (f(x,y+h)-f(x,y-h))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7)
    checks += 2
print(f'Central-difference gradient checks: {checks}/12 passed')

## 学生独立实现：rosenbrock_gradient(params)（通过）

学生原始代码见下一单元。该实现把此前的解析偏导正确转换为 Python，返回 `[df_dx, df_dy]`，`params` 的两个坐标解包顺序也正确。

在 Python 中已实际执行 14/14 项测试：六个点各两项中心有限差分共 12 项，另检验两个解析点 `grad([1,1]) == [0,0]` 及 `grad([-1,1]) == [-4,0]`。中心差分步长 `1e-6`；判定使用 `math.isclose(rel_tol=3e-8, abs_tol=4e-7)`。

**结论：该函数通过本轮验收。** 不需修改学生代码。

## docs/zh.md 格式修复

检查出中文精译版曾使用 `\[` / `\]` 块级公式定界符、五个无语言标签的代码围栏，以及一处仅供课程网站使用的 `figure` 标识。

已改为 GitHub 推荐的独立 `$$` 数学块、增加 `text` 代码标签，且把 `figure` 专用标签换成可阅读的文本与兼容说明。原文 17 个围栏块的**主体内容**保持一致，24 个标题保持一致，英文源文件不变。

**下一步**：根据官方课程 Build It / Step 2，自主实现 `GradientDescent.step(params, grads)`；确认数学方向和输入输出后再引入动量。

In [ ]:
def rosenbrock_gradient(params):
    # params = [x, y]
    # 返回 [df_dx, df_dy]
    x,y = params
    df_dx =  -2 * (1 - x) - 400 * x * (y - x ** 2)
    df_dy = 200 * ( y - x ** 2)

    return [df_dx, df_dy]

In [ ]:
import math

def rosenbrock_value(point):
    x, y = point
    return (1-x)**2 + 100*(y-x*x)**2

check_points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in check_points:
    gx, gy = rosenbrock_gradient([x,y])
    dx = (rosenbrock_value((x+h,y))-rosenbrock_value((x-h,y)))/(2*h)
    dy = (rosenbrock_value((x,y+h))-rosenbrock_value((x,y-h)))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dx',gx,dx)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dy',gy,dy)
    checks += 2
assert rosenbrock_gradient([1,1]) == [0,0]
assert rosenbrock_gradient([-1,1]) == [-4,0]
checks += 2
assert checks == 14
print('Rosenbrock gradient: 14/14 PASS')

## 下一项独立编码练习：GradientDescent（等待提交）

公式：`new_p = p - lr*g`（逐坐标计算）。

请只写如下类的 `__init__` 与 `step`，不要先看官方答案：

    class GradientDescent:
        def __init__(self, lr=0.001):
            pass

        def step(self, params, grads):
            pass

`params` 和 `grads` 是长度相同的序列，`step` 返回更新后的参数列表，不需要计算梯度。提交后通过多个维度、正负梯度和不同学习率进行测试。

In [ ]:
class GradientDescent:
        def __init__(self, lr=0.001):
            self.lr = lr

        def step(self, params, grads):
            new_p = params - self.lr * grads
            return new_p

## GradientDescent：学生首次代码提交与测试结果（需订正）

学生独立代码见上方单元；已完整保留原始写法。

**验收：`__init__` 正确；`step` 在要求的 Python 列表输入下失败，暂未通过。**

数学更新公式正确，但代码 `new_p = params - self.lr * grads` 是对 `list` 应用数组向量化运算。

导师实际运行输入 `params=[5.0,3.0]`, `grads=[4.0,-2.0]`, `lr=0.1`：

```text
TypeError: can't multiply sequence by non-int of type 'float'
```

同样的代码若输入为 `numpy.ndarray`，独立实际运行输出 `[4.6 3.2]`，与数学预期一致；但这不是本课程要求的 list 输入。

**待学生独立修复：** 使用 `zip(params, grads)` 逐坐标读取 `p,g`，计算 `p-self.lr*g` 并返回新列表；不提前提供最终完整类答案。接下来检验二维与多维输入、正负梯度、不同学习率。

该错误和订正要求必须保留，不因最后修正成功而删除原始版本。